In [1]:
library(Seurat)
library(reticulate)
library(anndata)
options(future.globals.maxSize = 1 * 1024^3)

xenium_name <- "Region2"
TABLE       <- "REGION2_TABLES"
xenium_dir <- "/Volumes/ProstateCancerEvoMain/raw/xenium/output-XETG00283__0037793__Region_2__20241204__160314"

cell_guide <- read.csv(file.path(
  "/Volumes/ProstateCancerEvoMain/dbs/Completed/AllRegions/CellTypes",
  paste0(xenium_name, ".raw.annotated.V2.guide.csv")
), stringsAsFactors=FALSE)

cell_guide

base_dir   <- file.path("/Volumes/ProstateCancerEvoMain/dbs/Ongoing", xenium_name, TABLE)
h5ad_file  <- file.path(
  base_dir,
  paste0(xenium_name, "_Xenium_Phen_HE_Integrated.Protein_PhenCycTable.V1.h5ad")
)
rds_file   <- file.path(
  base_dir,
  paste0(xenium_name, "_Xenium_Phen_HE_Integrated.Protein_PhenCycTable.V1.rds")
)

data <- read_h5ad(h5ad_file)

seu_phen <- CreateSeuratObject(
  counts    = t(as.matrix(data$X)),
  meta.data = data$obs
)

saveRDS(seu_phen, file = rds_file)

Loading required package: SeuratObject

Loading required package: sp

'SeuratObject' was built under R 4.4.1 but the current version is
4.4.2; it is recomended that you reinstall 'SeuratObject' as the ABI
for R may have changed


Attaching package: 'SeuratObject'


The following objects are masked from 'package:base':

    intersect, t



Attaching package: 'anndata'


The following object is masked from 'package:SeuratObject':

    Layers




X,cell_id,cell_type_L0,assigned_celltype_L0,leiden_L0
<int>,<chr>,<chr>,<chr>,<int>
0,aaaaealn-1,Ki67,tumor_cell_markers,0
1,aaaaeaol-1,FOXp3,CD8 T Cell,9
2,aaabgafi-1,pten,tumor_cell_markers,3
3,aaabhken-1,CD8,fibroblast_of_connective_tissue_of_glandular_part_of_prostate,6
4,aaabhmik-1,pten,,14
5,aaabhnhh-1,pten,CD8 and CD4 T Cell,2
6,aaabmedp-1,CD20,CD4 T Cell,8
7,aaadegpg-1,CD8,CD4 T Cell,8
8,aaadehdf-1,CD68,CD4 T Cell,8


Warning message:
"Data is of class matrix. Coercing to dgCMatrix."


In [2]:
"_____________________________________________________ Add Xenium from raw output folder _____________________________________________________________________"


adata_xenium <- LoadXenium(
  data.dir            = xenium_dir,    # path to folder with Xenium CSVs/parquets
  fov                 = "fov",         # name to assign to this field of view
  assay               = "Xenium",      # assay slot name
  #mols.qv.threshold   = 20,            # quality‐value cutoff for transcripts
  cell.centroids      = TRUE,          # load cell centroid coords
  molecule.coordinates= FALSE          # skip loading raw molecule pixels
)


[1] "_____________________________________________________ Add Xenium from raw output folder _____________________________________________________________________"

Genome matrix has multiple modalities, returning a list of matrices for this genome

Warning message:
"Feature names cannot have underscores ('_'), replacing with dashes ('-')"
Warning message:
"Feature names cannot have underscores ('_'), replacing with dashes ('-')"
Warning message:
"Feature names cannot have underscores ('_'), replacing with dashes ('-')"
Warning message:
"Feature names cannot have underscores ('_'), replacing with dashes ('-')"
Warning message:
"Feature names cannot have underscores ('_'), replacing with dashes ('-')"
Warning message:
"Feature names cannot have underscores ('_'), replacing with dashes ('-')"
Warning message:
"Feature names cannot have underscores ('_'), replacing with dashes ('-')"
Warning message:
"Feature names cannot have underscores ('_'), replacing with dashes ('-')"


In [3]:

"________________________________________________________ Change the names of PhenCyc with Xenium __________________________________________________________________"
adata_xenium
adata_phen <- seu_phen

new_names <- setNames(colnames(adata_xenium), colnames(adata_phen))
adata_phen <- RenameCells(adata_phen, new.names = new_names)


seu <- adata_xenium
DefaultAssay(seu) <- "Xenium"

seu[["PhenCyc"]] <- adata_phen[["RNA"]]


[1] "________________________________________________________ Change the names of PhenCyc with Xenium __________________________________________________________________"

An object of class Seurat 
6396 features across 299769 samples within 5 assays 
Active assay: Xenium (5101 features, 0 variable features)
 1 layer present: counts
 4 other assays present: BlankCodeword, ControlCodeword, ControlProbe, GenomicControl
 1 spatial field of view present: fov

In [4]:
" _____________________________________________________ Define Cell Guides Previous Annotations ____________________________________________________________"

rownames(cell_guide) <- cell_guide$cell_id
rownames(cell_guide)

cell_guide$cell_id <- NULL

all(rownames(seu@meta.data) %in% rownames(cell_guide))

seu <- AddMetaData(
  object   = seu,
  metadata = cell_guide
)

[1] " _____________________________________________________ Define Cell Guides Previous Annotations ____________________________________________________________"

[1] "aaaaealn-1" "aaaaeaol-1" "aaabgafi-1" "aaabhken-1" "aaabhmik-1"
    [6] "aaabhnhh-1" "aaabmedp-1" "aaadegpg-1" "aaadehdf-1" "aaadfabf-1"
   [11] "aaadomne-1" "aaaeaidk-1" "aaaecpdd-1" "aaaegfkh-1" "aaaegglm-1"
   [16] "aaaeglng-1" "aaaenmbe-1" "aaaeodbm-1" "aaafiflp-1" "aaafoggm-1"
   [21] "aaagalcm-1" "aaagjbio-1" "aaagpdep-1" "aaahdkhb-1" "aaahedid-1"
   [26] "aaahgbcj-1" "aaahinld-1" "aaahnaip-1" "aaahokhh-1" "aaaibmik-1"
   [31] "aaaihhbe-1" "aaaihoel-1" "aaaijjng-1" "aaaikggi-1" "aaainjig-1"
   [36] "aaaiodoj-1" "aaaipgcm-1" "aaaiphgf-1" "aaajddme-1" "aaajepeb-1"
   [41] "aaajfjbm-1" "aaajhjba-1" "aaajimja-1" "aaajlikd-1" "aaajpanc-1"
   [46] "aaakdnnh-1" "aaakjikp-1" "aaaknajb-1" "aaakondd-1" "aaakpfhj-1"
   [51] "aaalgocc-1" "aaaljjni-1" "aaaloina-1" "aaalpkkg-1" "aaamedcf-1"
   [56] "aaamkbng-1" "aaamlefj-1" "aaammgin-1" "aaamnblg-1" "aaamomog-1"
   [61] "aaanbile-1" "aaanengo-1" "aaanhmkj-1" "aaanipof-1" "aaanncma-1"
   [66] "aaanohgb-1" "aaaohadh-1" "aaaohdfo-1" "aaaokabd-1" "aaaokhcj-1"
   [71] "aaaolbhi-1" "aaaonkea-1" "aaapbmig-1" "aaapebkp-1" "aaapeoce-1"
   [76] "aaaphkfl-1" "aabaeohl-1" "aabaipdm-1" "aabananc-1" "aabanihd-1"
   [81] "aabappch-1" "aabbdbee-1" "aabbfaef-1" "aabbgiip-1" "aabbgogp-1"
   [86] "aabbjlac-1" "aabbmdkg-1" "aabbmndg-1" "aabceoll-1" "aabcjkgh-1"
   [91] "aabclnjm-1" "aabcnagi-1" "aabdinci-1" "aabdndce-1" "aabdodaa-1"
   [96] "aabdopcb-1" "aabeajbj-1" "aabefepk-1" "aabeopfc-1" "aabfhopp-1"
  [101] "aabfnhla-1" "aabfpogd-1" "aabgakba-1" "aabgbmmh-1" "aabgmkcc-1"
  [106] "aabgnpdg-1" "aabgpelf-1" "aabhcacp-1" "aabhdojk-1" "aabhehfp-1"
  [111] "aabheiem-1" "aabhgjmp-1" "aabhldgl-1" "aabhlhmk-1" "aabidjcc-1"
  [116] "aabiefcp-1" "aabijghc-1" "aabjgkdo-1" "aabjhegb-1" "aabjljgh-1"
  [121] "aabjoklk-1" "aabjolfg-1" "aabjpkkj-1" "aabjpppi-1" "aabkfjjf-1"
  [126] "aabkjdik-1" "aabkkdpm-1" "aabkmfci-1" "aabknibd-1" "aabkobci-1"
  [131] "aabkpnag-1" "aabkpncb-1" "aablbade-1" "aablblkl-1" "aablflgf-1"
  [136] "aablhhdf-1" "aablhhjd-1" "aablmdbh-1" "aabmaidc-1" "aabmiepl-1"
  [141] "aabmmfga-1" "aabmnkjm-1" "aabnbjhg-1" "aabncbhd-1" "aabnfjje-1"
  [146] "aabnfkoj-1" "aaboaond-1" "aaboejdb-1" "aabogohn-1" "aabojehl-1"
  [151] "aabpomom-1" "aacaolnn-1" "aacapkpd-1" "aacbfiap-1" "aacbiafl-1"
  [156] "aaccaiik-1" "aacchmac-1" "aaccijoj-1" "aaccopmc-1" "aaccpdjl-1"
  [161] "aacdhjin-1" "aacdjfoa-1" "aacdolej-1" "aacdpegd-1" "aaceaehb-1"
  [166] "aacecdja-1" "aacedajc-1" "aaceimag-1" "aacembfn-1" "aacfadok-1"
  [171] "aacfakgb-1" "aacfaome-1" "aacfdkpm-1" "aacfidki-1" "aacfkmmk-1"
  [176] "aacgchia-1" "aacgeonb-1" "aacgkoco-1" "aacgpjce-1" "aachafjo-1"
  [181] "aachccba-1" "aachdbco-1" "aachdklh-1" "aacheafm-1" "aachfnml-1"
  [186] "aachhdie-1" "aachimoi-1" "aachjogh-1" "aachkhjj-1" "aachkpji-1"
  [191] "aachpjkl-1" "aacibhbh-1" "aacieppf-1" "aacifohj-1" "aacigapf-1"
  [196] "aacilipk-1" "aacjaafc-1" "aacjeclo-1" "aacjgmce-1" "aacjjfcg-1"
  [201] "aacjkcmd-1" "aacjngnn-1" "aackbjmn-1" "aackbljc-1" "aackbnbf-1"
  [206] "aackbpal-1" "aackjkmf-1" "aackkmlf-1" "aackmmpe-1" "aackmoak-1"
  [211] "aaclafcf-1" "aaclbgdk-1" "aacleigp-1" "aaclgncc-1" "aaclkafm-1"
  [216] "aacllled-1" "aaclnecc-1" "aacmclic-1" "aacmelbp-1" "aacmfegh-1"
  [221] "aacmhpan-1" "aacmlkpn-1" "aacmnegj-1" "aacmnhgc-1" "aacmolpc-1"
  [226] "aacmpkga-1" "aacneapd-1" "aacnfgim-1" "aacngonp-1" "aacnibcj-1"
  [231] "aacnicjl-1" "aacnjngp-1" "aacnnocd-1" "aacnopao-1" "aacnppel-1"
  [236] "aacogdlf-1" "aacoibnk-1" "aacoijkm-1" "aacokdpm-1" "aacomdbo-1"
  [241] "aacoofll-1" "aacpbgnj-1" "aacpdcjp-1" "aacplbko-1" "aacpmabl-1"
  [246] "aacpmdpl-1" "aadaabdi-1" "aadadmga-1" "aadagcdb-1" "aadahhmd-1"
  [251] "aadalenj-1" "aadanfje-1" "aadbbemj-1" "aadbhmjh-1" "aadbifbi-1"
  [256] "aadbkden-1" "aadbkgfn-1" "aadbkodo-1" "aadbmkdp-1" "aadcajdb-1"
  [261] "aadccfdj-1" "aadclbdf-1" "aadcldpg-1" "aadcljmi-1" "aaddalba-1"
  [266] "aaddgaep-1" "aaddiiog-1" "aaddompc-1" "aadeagff-1" "aadeebij-1"
  [271] "aadefgej-1" "aadefmjo-1" "aadeiank-1" "aadendal-1" "a

[1] TRUE

In [5]:

"________________________________________________________ Concat PhenCyc + Xenium in one Obj and Preprocess [PCA + Variable Genes + Scaling] __________________________________________________________________"

for (assay in c("Xenium","PhenCyc")) {
  DefaultAssay(seu) <- assay
  seu <- NormalizeData(seu, verbose = FALSE)
  
  seu <- FindVariableFeatures(seu, selection.method = "vst", nfeatures = 2000, verbose = FALSE)
  
  seu <- ScaleData(seu, features = VariableFeatures(seu), verbose = FALSE)
  
  
  seu <- RunPCA(seu,
                features = VariableFeatures(seu),
                reduction.name = paste0("pca_", assay),
                verbose = FALSE)
}



[1] "________________________________________________________ Concat PhenCyc + Xenium in one Obj and Preprocess [PCA + Variable Genes + Scaling] __________________________________________________________________"

Warning message in svd.function(A = t(x = object), nv = npcs, ...):
"You're computing too large a percentage of total singular values, use a standard svd instead."
Warning message:
"Key 'PC_' taken, using 'pcaphencyc_' instead"


In [ ]:
"________________________________________________________ Holy - WNN Part  __________________________________________________________________"

seu <- FindMultiModalNeighbors(
  object = seu,
  reduction.list    = list("pca_Xenium", "pca_PhenCyc"),
  dims.list         = list(1:20,       1:20),
  modality.weight.name = "PhenCyc.weight"
)

[1] "________________________________________________________ Holy - WNN Part  __________________________________________________________________"

Calculating cell-specific modality weights

Finding 20 nearest neighbors for each modality.

Calculating kernel bandwidths

Warning message in FindMultiModalNeighbors(object = seu, reduction.list = list("pca_Xenium", :
"The number of provided modality.weight.name is not equal to the number of modalities. Xenium.weight PhenCyc.weight are used to store the modality weights"
Finding multimodal neighbors

Warning message:
"Caught FutureInterruptError. Canceling all iterations ..."


In [ ]:
"________________________________________________________ Holy - WNN Part  __________________________________________________________________"

seu <- RunUMAP(seu, nn.name = "weighted.nn", reduction.name = "wnn.umap",
               reduction.key = "wnnUMAP_")

seu <- FindClusters(seu, graph.name = "wsnn", algorithm = 3, resolution = 0.5)



In [ ]:
"________________________________________________________ Annotating Clusters  __________________________________________________________________"


# Assuming your clusters are stored in seu$seurat_clusters
Idents(seu) <- "seurat_clusters"

# 3. Find markers using only the Xenium assay
markers_xenium <- FindAllMarkers(
  object          = seu,
  assay           = "Xenium",        # explicitly use Xenium assay
  slot            = "data",          # use normalized data slot
  only.pos        = TRUE,
  min.pct         = 0.25,
  logfc.threshold = 0.25
)

library(tidyverse)   # loads dplyr + ggplot2 + etc.

# Inspect the top 5 markers per cluster
top5 <- markers_xenium %>% 
  group_by(cluster) %>% 
  slice_max(order_by = avg_log2FC, n = 5)
print(top5)



In [ ]:
"________________________________________________________ Plot Clusters  __________________________________________________________________"

p_clusters <- DimPlot(
  object    = seu,
  reduction = "wnn.umap",
  group.by  = "seurat_clusters",  # or replace with "wsnn" if that's your column
  label     = TRUE
) + ggtitle("WNN‐UMAP Clusters (resolution = 0.5)")

p_clusters



In [ ]:

"________________________________________________________ Annotating Clusters by Guide  __________________________________________________________________"

# 1. Coerce to character so we can fill in NAs
seu$assigned_celltype_L0 <- as.character(seu$assigned_celltype_L0)
# 2. Fill missing/empty
seu$assigned_celltype_L0[is.na(seu$assigned_celltype_L0) | seu$assigned_celltype_L0 == ""] <- "Unknown"
# 3. Turn back into a factor (optional: specify levels if you want a custom order)
seu$assigned_celltype_L0 <- factor(seu$assigned_celltype_L0)

# 4. Now DimPlot will work
p_cell_type_all <- DimPlot(
  object    = seu,
  reduction = "wnn.umap",
  group.by  = "assigned_celltype_L0",
  label     = TRUE,
  label.size= 4,
  raster    = FALSE
)
print(p_cell_type_all)

#seu$NF_H_vs_Others <- ifelse(
#  seu$cell_type == "NF-H",   # replace with your exact CD8 label
#  "NF-H",
#  "Other"
#)

#p_phen_assigned <- DimPlot(
#  seu,
#  reduction = "wnn.umap", 
#  group.by  = "NF_H_vs_Others",
#  cols      = c("NF-H" = "red", "Other" = "blue"),
#  pt.size   = 0.5
#) 
#p_phen_assigned



In [ ]:
"________________________________________________________ CrossTab Set   __________________________________________________________________"

library(pheatmap)

meta <- seu@meta.data

ct_tab <- table(
  CellType          = meta$cell_type,
  AssignedCellType  = meta$assigned_celltype
)

# 4. (Optional) Normalize by row to show proportions
 ct_tab <- prop.table(ct_tab, margin = 2)

# 5. Plot heatmap
pheatmap(
  mat            = ct_tab,
  cluster_rows   = FALSE,
  cluster_cols   = FALSE,
  display_numbers= TRUE,
  fontsize       = 10,
  angle_col      = 45,
  main           = "Overlap: cell_type vs assigned_celltype"
)


rds_target_file   <- file.path(
  "/Volumes/ProstateCancerEvoMain/dbs/Completed/AllRegions/MMI_WNN",
  paste0(xenium_name, ".raw.annotated.V2.seu.wnn.rds")
)

rds_target_file
#saveRDS(seu, file = rds_target_file)
